In [ ]:
# ==========================================

# 3.1 — Definição do dataset municipal para clustering

# ==========================================



df_ml = df_master_cidades.copy()



print("Total de municípios na base municipal:", df_ml.shape[0])

display(df_ml[['CO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_PROVA', 'SG_UF_PROVA']].head())

# ==========================================

# 3.2 — Seleção de features para clustering (inclui MEDIA_GERAL)

# ==========================================



features = [

    'PERC_SEM_INTERNET',

    'PERC_SEM_PC',

    'PERC_ESCOLA_PUBLICA',

    'TAXA_ABSTENCAO',

    'MEDIA_GERAL'

]



# Dataset numérico final do modelo

X = df_ml[features].dropna().copy()



# Garantir alinhamento entre df_ml e X

df_ml = df_ml.loc[X.index].copy()



print("Formato do dataset de ML:", X.shape)

print("Features usadas:", features)

# ==========================================

# 3.4 — Split treino/teste + escolha do melhor K

# ==========================================



from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

from sklearn.cluster import KMeans

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import silhouette_score, davies_bouldin_score

import matplotlib.pyplot as plt



# Split para checar estabilidade do clustering

X_train, X_test = train_test_split(X, test_size=0.25, random_state=42)



print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")



k_values = range(2, 11)



inertias = []

sil_train = []

sil_test = []

db_train = []

db_test = []



for k in k_values:

    pipeline = Pipeline(steps=[

        ("scaler", StandardScaler()),

        ("kmeans", KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=42))

    ])



    pipeline.fit(X_train)



    labels_train = pipeline.named_steps["kmeans"].labels_

    labels_test = pipeline.predict(X_test)



    Xtr_scaled = pipeline.named_steps["scaler"].transform(X_train)

    Xte_scaled = pipeline.named_steps["scaler"].transform(X_test)



    inertias.append(pipeline.named_steps["kmeans"].inertia_)

    sil_train.append(silhouette_score(Xtr_scaled, labels_train))

    sil_test.append(silhouette_score(Xte_scaled, labels_test))

    db_train.append(davies_bouldin_score(Xtr_scaled, labels_train))

    db_test.append(davies_bouldin_score(Xte_scaled, labels_test))



# Visualizações para decidir o K

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fig.suptitle("Seleção de K (Treino vs Teste): Elbow / Silhouette / Davies-Bouldin", fontweight="bold")



axes[0].plot(list(k_values), inertias, marker="o")

axes[0].set_title("Elbow (Inércia/SSE)")

axes[0].set_xlabel("K")

axes[0].set_ylabel("SSE")



axes[1].plot(list(k_values), sil_train, marker="o", label="Treino")

axes[1].plot(list(k_values), sil_test, marker="o", label="Teste")

axes[1].set_title("Silhouette (maior é melhor)")

axes[1].set_xlabel("K")

axes[1].set_ylabel("Score")

axes[1].legend()



axes[2].plot(list(k_values), db_train, marker="o", label="Treino")

axes[2].plot(list(k_values), db_test, marker="o", label="Teste")

axes[2].set_title("Davies-Bouldin (menor é melhor)")

axes[2].set_xlabel("K")

axes[2].set_ylabel("Índice DB")

axes[2].legend()



plt.tight_layout()

plt.show()

In [ ]:
# ==========================================

# 3.5 — Treinamento final + criação da coluna CLUSTER

# ==========================================



from sklearn.metrics import calinski_harabasz_score



K_FINAL = 4  # <-- Ajuste com base nos gráficos da etapa 3.4



pipeline_final = Pipeline(steps=[

    ("scaler", StandardScaler()),

    ("kmeans", KMeans(n_clusters=K_FINAL, init="k-means++", n_init=30, random_state=42))

])



pipeline_final.fit(X_train)



# Métricas finais (estabilidade)

Xtr_scaled = pipeline_final.named_steps["scaler"].transform(X_train)

Xte_scaled = pipeline_final.named_steps["scaler"].transform(X_test)



labels_train = pipeline_final.named_steps["kmeans"].labels_

labels_test = pipeline_final.predict(X_test)



print("=== MÉTRICAS FINAIS ===")

print(f"Silhouette (Treino): {silhouette_score(Xtr_scaled, labels_train):.3f}")

print(f"Silhouette (Teste):  {silhouette_score(Xte_scaled, labels_test):.3f}")

print(f"Davies-Bouldin (Treino): {davies_bouldin_score(Xtr_scaled, labels_train):.3f}")

print(f"Davies-Bouldin (Teste):  {davies_bouldin_score(Xte_scaled, labels_test):.3f}")

print(f"Calinski-Harabasz (Treino): {calinski_harabasz_score(Xtr_scaled, labels_train):.1f}")

print(f"Calinski-Harabasz (Teste):  {calinski_harabasz_score(Xte_scaled, labels_test):.1f}")



# Aplicar clusters para TODOS os municípios (base completa)

df_ml["CLUSTER"] = pipeline_final.predict(X).astype(int)



display(df_ml[['NO_MUNICIPIO_PROVA', 'SG_UF_PROVA', 'TOTAL_INSCRITOS', 'MEDIA_GERAL', 'TAXA_ABSTENCAO', 'CLUSTER']].head())


In [ ]:
# ==========================================

# 3.6 — Interpretação: perfil médio por cluster

# ==========================================



import seaborn as sns



perfil_clusters = df_ml.groupby("CLUSTER")[features].mean().round(2)

perfil_clusters["QT_MUNICIPIOS"] = df_ml["CLUSTER"].value_counts().sort_index()



display(perfil_clusters)



plt.figure(figsize=(12, 6))

sns.heatmap(perfil_clusters[features], annot=True, cmap="viridis", fmt=".1f")

plt.title("Perfil Médio dos Clusters (Municípios)", fontweight="bold")

plt.ylabel("Cluster")

plt.xlabel("Indicadores")

plt.show()

In [ ]:
# ==========================================

# 3.8 — GRÁFICO COMPARATIVO FINAL

# Média Geral (MEDIA_GERAL) por Cluster

# ==========================================



import matplotlib.pyplot as plt

import seaborn as sns

import numpy as np



sns.set_theme(style="whitegrid")



# Tabela agregada: média e desvio padrão da média geral por cluster

df_comp = df_ml.groupby("CLUSTER").agg(

    MEDIA_GERAL_MEDIA=("MEDIA_GERAL", "mean"),

    MEDIA_GERAL_STD=("MEDIA_GERAL", "std"),

    QT_MUNICIPIOS=("MEDIA_GERAL", "count")

).reset_index()



# Ordena do pior para o melhor (ou inverso)

df_comp = df_comp.sort_values("MEDIA_GERAL_MEDIA", ascending=True)



plt.figure(figsize=(12, 6))

ax = sns.barplot(

    data=df_comp,

    x="CLUSTER",

    y="MEDIA_GERAL_MEDIA",

    palette="viridis"

)



# Barras de erro (STD)

ax.errorbar(

    x=np.arange(df_comp.shape[0]),

    y=df_comp["MEDIA_GERAL_MEDIA"],

    yerr=df_comp["MEDIA_GERAL_STD"],

    fmt="none",

    ecolor="black",

    elinewidth=2,

    capsize=6

)



# Labels com média e n

for i, row in df_comp.reset_index(drop=True).iterrows():

    ax.text(

        i,

        row["MEDIA_GERAL_MEDIA"] + 3,

        f"{row['MEDIA_GERAL_MEDIA']:.1f}\n(n={int(row['QT_MUNICIPIOS'])})",

        ha="center",

        va="bottom",

        fontsize=10,

        fontweight="bold"

    )



plt.title("Comparativo Final: Desempenho Médio (MEDIA_GERAL) por Cluster", fontsize=14, fontweight="bold")

plt.xlabel("Cluster (K-Means)")

plt.ylabel("Média Geral do Município (0 a 1000)")

plt.tight_layout()

plt.show()

In [ ]:
# ==========================================

# 3.9 — GRÁFICO FINAL (RADAR)

# Perfil médio das features por cluster

# ==========================================



import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler



# Features usadas no clustering (as mesmas do seu KMeans)

features = [

    'PERC_SEM_INTERNET',

    'PERC_SEM_PC',

    'PERC_ESCOLA_PUBLICA',

    'TAXA_ABSTENCAO',

    'MEDIA_GERAL'

]



# Médias por cluster

perfil = df_ml.groupby("CLUSTER")[features].mean().reset_index()



# Normalização 0-1 para radar (senão MEDIA_GERAL domina)

scaler = MinMaxScaler()

perfil_scaled = perfil.copy()

perfil_scaled[features] = scaler.fit_transform(perfil[features])



# Preparar radar

labels = features

num_vars = len(labels)



angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False).tolist()

angles += angles[:1]  # fecha o círculo



plt.figure(figsize=(10, 8))

ax = plt.subplot(111, polar=True)



# Plot por cluster

for _, row in perfil_scaled.iterrows():

    values = row[labels].tolist()

    values += values[:1]

    ax.plot(angles, values, linewidth=2, label=f"Cluster {int(row['CLUSTER'])}")

    ax.fill(angles, values, alpha=0.10)



ax.set_thetagrids(np.degrees(angles[:-1]), labels)

plt.title("Perfil Comparativo dos Clusters (Radar - Valores Normalizados 0 a 1)", fontsize=14, fontweight="bold")

plt.legend(loc="upper right", bbox_to_anchor=(1.25, 1.10))

plt.tight_layout()

plt.show()

df_ml.groupby("CLUSTER")[['PERC_SEM_INTERNET','PERC_SEM_PC','PERC_ESCOLA_PUBLICA','TAXA_ABSTENCAO','MEDIA_GERAL']].mean()

In [ ]:
# ==========================================
# MODELAGEM COM K-MEDOIDS E COMPARAÇÃO DE MODELOS
# ==========================================
!pip install scikit-learn-extra
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score, davies_bouldin_score
import time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score


# Definindo o intervalo de clusters a serem testados (de 2 a 8 para otimizar tempo)
k_values = range(2, 9)
inertias_kmed = []
sil_kmed = []

# Aplicando o Scaler no X_train uma única vez fora do loop para otimizar o processamento
scaler_teste = StandardScaler()
X_train_scaled_teste = scaler_teste.fit_transform(X_train)

for k in k_values:
    # Treinando K-Medoids para o valor atual de K
    kmed = KMedoids(n_clusters=k, init='k-medoids++', random_state=42)
    kmed.fit(X_train_scaled_teste)

    # Salvando a Inércia (Elbow)
    # No K-Medoids, a inércia é a soma das distâncias dos pontos ao seu medoide mais próximo
    inertias_kmed.append(kmed.inertia_)

    # Calculando e salvando o Silhouette Score
    labels = kmed.labels_
    sil_score = silhouette_score(X_train_scaled_teste, labels)
    sil_kmed.append(sil_score)


In [ ]:
# ==========================================
# PLOTAGEM DOS GRÁFICOS (ELBOW E SILHOUETTE)
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise de K para K-Medoids: Elbow e Silhouette", fontsize=14, fontweight="bold")

# Gráfico 1: Elbow Method (Inércia)
axes[0].plot(list(k_values), inertias_kmed, marker='o', color='#d95f02', linewidth=2)
axes[0].set_title("Método do Cotovelo (Elbow) - K-Medoids", fontsize=12)
axes[0].set_xlabel("Número de Clusters (K)")
axes[0].set_ylabel("Inércia (Soma das Distâncias ao Medoide)")
axes[0].grid(True, linestyle='--', alpha=0.6)

# Gráfico 2: Silhouette Score
axes[1].plot(list(k_values), sil_kmed, marker='s', color='#7570b3', linewidth=2)
axes[1].set_title("Silhouette Score - K-Medoids", fontsize=12)
axes[1].set_xlabel("Número de Clusters (K)")
axes[1].set_ylabel("Score (Quanto mais próximo de 1, melhor)")
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


# Vamos usar o mesmo número de cluster
K_FINAL = 4

# 1. Pipeline do K-Medoids
pipeline_kmedoids = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("kmedoids", KMedoids(n_clusters=K_FINAL, init='k-medoids++', random_state=42))
])

# Medindo o tempo de treinamento do K-Medoids
start_time_kmed = time.time()
pipeline_kmedoids.fit(X_train)
tempo_kmedoids = time.time() - start_time_kmed

# Previsões K-Medoids no Teste
Xte_scaled = pipeline_kmedoids.named_steps["scaler"].transform(X_test)
labels_test_kmedoids = pipeline_kmedoids.predict(X_test)

# Métricas do K-Medoids
sil_kmedoids = silhouette_score(Xte_scaled, labels_test_kmedoids)
db_kmedoids = davies_bouldin_score(Xte_scaled, labels_test_kmedoids)


# 2. Resgatando as métricas e tempos do K-Means para comparar
start_time_kmeans = time.time()
pipeline_final.fit(X_train)
tempo_kmeans = time.time() - start_time_kmeans

labels_test_kmeans = pipeline_final.predict(X_test)
sil_kmeans = silhouette_score(Xte_scaled, labels_test_kmeans)
db_kmeans = davies_bouldin_score(Xte_scaled, labels_test_kmeans)

# 3. Construção da Matriz Comparativa

df_comparacao = pd.DataFrame({
    'Modelo': ['K-Means', 'K-Medoids'],
    'Silhouette Score (Maior = Melhor)': [round(sil_kmeans, 3), round(sil_kmedoids, 3)],
    'Davies-Bouldin (Menor = Melhor)': [round(db_kmeans, 3), round(db_kmedoids, 3)],
    'Tempo de Treinamento (Segundos)': [round(tempo_kmeans, 4), round(tempo_kmedoids, 4)]
})

display(df_comparacao)


# 4. Adicionando a coluna do K-Medoids ao dataset original para inferência
df_ml["CLUSTER_KMEDOIDS"] = pipeline_kmedoids.predict(X).astype(int)

# Mostrando as diferenças de classificação entre os modelos
print("\n>>> Amostra das Diferenças de Clusterização <<<")
display(df_ml[['NO_MUNICIPIO_PROVA', 'MEDIA_GERAL', 'CLUSTER', 'CLUSTER_KMEDOIDS']].head(10))